[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: entornos en GitHub Actions

*Nota para quien imparte el curso: el cuaderno 12 describe los entornos, pero no incluye una actividad con enunciado. Los ejercicios se derivan de las afirmaciones del texto («los *workflows* de esos cuadernos referencian el entorno con la clave `environment`») y de la nota de instructor.*

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

**Disponibilidad.** Los entornos con reglas de protección están disponibles en repositorios públicos y, en los privados, según el plan de la cuenta. Las celdas con etiqueta fallan con un mensaje claro si el plan no los admite.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Diseñar la configuración de los entornos `cloudrun` y `linux`

El curso usa dos entornos protegidos. Cada uno debe limitar los despliegues a la rama `main`. Opcionalmente puede requerir la aprobación de una persona revisora y esperar unos minutos antes de ejecutar.

La *API* de *GitHub* recibe un objeto *JSON*. La siguiente función lo construye y comprueba los valores: el tiempo de espera es un entero entre 0 y 43200 minutos, y las personas revisoras se identifican por su número de usuario.

La siguiente celda definirá `configuracion_entorno(espera, revisores)`, que devuelve el cuerpo de la petición `PUT` de un entorno con política de ramas personalizada, validando los argumentos.

In [ ]:
import json

def configuracion_entorno(espera=0, revisores=()):
    if not isinstance(espera, int) or not 0 <= espera <= 43200:
        raise ValueError("espera debe ser un entero entre 0 y 43200 minutos")
    cuerpo = {
        "wait_timer": espera,
        "deployment_branch_policy": {"protected_branches": False, "custom_branch_policies": True},
    }
    if revisores:
        cuerpo["reviewers"] = [{"type": "User", "id": int(identificador)} for identificador in revisores]
    return cuerpo

La siguiente celda probará la función con tres casos: sin opciones, con una espera de 5 minutos y con una persona revisora de número de usuario 1 (un número de ejemplo). También comprobará que un valor fuera de rango se rechaza.

In [ ]:
basica = configuracion_entorno()
con_espera = configuracion_entorno(espera=5)
con_revisor = configuracion_entorno(revisores=[1])
print(json.dumps(con_revisor, indent=2))

assert basica["deployment_branch_policy"]["custom_branch_policies"] is True
assert con_espera["wait_timer"] == 5
assert con_revisor["reviewers"] == [{"type": "User", "id": 1}]
try:
    configuracion_entorno(espera=50000)
except ValueError:
    print("Rechazado: espera fuera de rango")
else:
    raise AssertionError("debía fallar")

La siguiente celda creará los dos entornos en el repositorio actual y añadirá a cada uno una política que solo permite la rama `main`. Para exigir revisión de una persona se añade `reviewers` al cuerpo, con el número que devuelve `gh api user --jq .id`.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
REPO="$(gh repo view --json nameWithOwner --jq .nameWithOwner)"
for ENTORNO in cloudrun linux; do
  gh api --method PUT "repos/${REPO}/environments/${ENTORNO}" --input - <<'JSON' > /dev/null
{"wait_timer": 0, "deployment_branch_policy": {"protected_branches": false, "custom_branch_policies": true}}
JSON
  gh api --method POST "repos/${REPO}/environments/${ENTORNO}/deployment-branch-policies" \
    --field name=main --field type=branch > /dev/null
done
gh api "repos/${REPO}/environments" --jq '.environments[] | {name, ramas: .deployment_branch_policy}'

**Salida esperada.** Dos objetos, uno con `"name": "cloudrun"` y otro con `"name": "linux"`, ambos con `"custom_branch_policies": true`. En `Settings` → `Environments`, cada entorno muestra la regla `Selected branches: main`.

## Ejercicio 2. Comprobar que los workflows referencian los entornos

Los *workflows* de despliegue deben asignar a su *job* el entorno correspondiente con la clave `environment`: `cloudrun` en el cuaderno 16 y `linux` en el 17.

La siguiente celda leerá los dos *workflows* de despliegue y comprobará con `assert` que el *job* `desplegar` de cada uno declara el entorno esperado. La clave `environment` puede ser un texto o un objeto con `name`.

In [ ]:
ESPERADO = {"deploy-cloud-run.yml": "cloudrun", "deploy-linux-ssh.yml": "linux"}
for archivo, entorno in ESPERADO.items():
    wf = u.cargar_workflow(RAIZ / ".github" / "workflows" / archivo)
    declarado = wf["jobs"]["desplegar"]["environment"]
    nombre = declarado["name"] if isinstance(declarado, dict) else declarado
    print(f"{archivo:25} -> {nombre}")
    assert nombre == entorno

## Ejercicio 3. Listar las variables y los secretos de cada entorno

Cada entorno guarda sus propias variables y secretos. Obtener, a partir de los *workflows*, qué nombres hay que definir en cada uno.

La siguiente celda extraerá con una expresión regular todos los nombres `vars.X` y `secrets.X` de cada *workflow* de despliegue y comprobará que contienen los nombres que piden los cuadernos 16 y 17. Se excluye `GITHUB_TOKEN`, que *GitHub* crea solo.

In [ ]:
import re

REQUERIDOS = {
    "deploy-cloud-run.yml": {"vars": {"GCP_PROJECT_ID", "GCP_REGION", "AR_REPOSITORY", "CLOUD_RUN_SERVICE",
                                       "GCP_WIF_PROVIDER", "GCP_DEPLOY_SA", "GCP_RUNTIME_SA"}, "secrets": set()},
    "deploy-linux-ssh.yml": {"vars": {"SSH_DEPLOY_DIR", "DEPLOY_RUNNER"},
                              "secrets": {"SSH_HOST", "SSH_USER", "SSH_PORT", "SSH_PRIVATE_KEY", "SSH_KNOWN_HOSTS"}},
}
for archivo, requerido in REQUERIDOS.items():
    texto = (RAIZ / ".github" / "workflows" / archivo).read_text(encoding="utf-8")
    encontrados = {"vars": set(), "secrets": set()}
    for contexto, nombre in re.findall(r"\b(vars|secrets)\.([A-Z][A-Z0-9_]*)", texto):
        encontrados[contexto].add(nombre)
    encontrados["secrets"].discard("GITHUB_TOKEN")
    print(archivo)
    for contexto in ("vars", "secrets"):
        print(f"  {contexto:8}", sorted(encontrados[contexto]))
        assert requerido[contexto] <= encontrados[contexto]

La siguiente celda definirá las variables del entorno `cloudrun` desde la terminal. Los valores salen de variables de entorno de la sesión (`GCP_PROJECT_ID`, `GCP_REGION` y las demás), nunca se escriben en el cuaderno. La celda se detiene si falta alguna.

In [ ]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
for NOMBRE in GCP_PROJECT_ID GCP_REGION AR_REPOSITORY CLOUD_RUN_SERVICE GCP_WIF_PROVIDER GCP_DEPLOY_SA GCP_RUNTIME_SA; do
  : "${!NOMBRE:?Defina ${NOMBRE} en la sesión antes de continuar}"
  gh variable set "${NOMBRE}" --env cloudrun --body "${!NOMBRE}"
done
gh variable list --env cloudrun

**Salida esperada.** Siete filas, una por variable, con sus valores (no son secretos). Los valores concretos dependen del proyecto de cada persona.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023-2026.</p>